# Scaled training and manual quality review

Historical research source, original zero-based cells **47?55**. Outputs have been cleared.

These cells retain the original global state, Colab paths, manual annotations, and checkpoint assumptions. This notebook is not standalone; earlier state and external artifacts may be required. Some repair branches are alternatives. Read [the migration guide](../../docs/migration.md) before selecting cells to run. For a new baseline, use [the package workflow](../amharic_asr.ipynb).


## Original source cell 47

Position 48 in the original notebook; original code comments may use different numbering.


In [ ]:

# ============================================================
# CELL 43 - SCALED DATASET: 64 TRAIN / 16 HELD-OUT
#
# Reuse the 8 previously trained recordings.
# Select fresh held-out speakers.
# Prefer shorter recordings and diverse speakers.
#
# No GPU training or checkpoint modifications.
# ============================================================

import json
import random
import statistics
from collections import defaultdict, Counter
from pathlib import Path
from IPython.display import Audio, display

SEED_43 = 20261019
TRAIN_TARGET = 64
VAL_TARGET = 16
VAL_SPEAKERS = 8
VAL_PER_SPEAKER = 2

EXP43_DIR = DATA_DIR / "scaled_amharic_64_16_v1"
EXP43_DIR.mkdir(parents=True, exist_ok=True)
EXP43_SELECTION = EXP43_DIR / "selection.json"

OLD_SPLIT = (
    DATA_DIR
    / "short_multirecording_head_only_v2"
    / "split.json"
)

assert OLD_SPLIT.is_file()

previous_split = json.loads(
    OLD_SPLIT.read_text(encoding="utf-8")
)

carryover_ids = previous_split["train_ids"]
assert len(carryover_ids) == 8

by_id_43 = {r["id"]: r for r in train_records}
assert all(rid in by_id_43 for rid in carryover_ids)

# Exclude every speaker previously involved in
# our small experiments from the NEW validation set.
prior_ids = set(
    previous_split["train_ids"]
    + previous_split["val_ids"]
)

old_long_selection = (
    DATA_DIR
    / "head_only_multirecording_v1"
    / "selection.json"
)

if old_long_selection.exists():
    old = json.loads(
        old_long_selection.read_text(encoding="utf-8")
    )
    prior_ids.update(old["train_ids"])
    prior_ids.update(old["val_ids"])

prior_speakers = {
    by_id_43[rid]["speaker_id"]
    for rid in prior_ids
    if rid in by_id_43
}

KERNELS_43 = (10, 3, 3, 3, 3, 2, 2)
STRIDES_43 = (5, 2, 2, 2, 2, 2, 2)

def count_frames_43(samples):
    for kernel, stride in zip(KERNELS_43, STRIDES_43):
        samples = (samples - kernel) // stride + 1
    return samples

def eligible_43(record, max_duration, allow_digits):
    duration = float(record["duration_seconds"])
    text = record["transcription"]

    if not (4.0 <= duration <= max_duration):
        return False

    if not record.get("speaker_id"):
        return False

    if not allow_digits and any(c.isdigit() for c in text):
        return False

    path = Path(AUDIO_DIR) / record["audio_file"]
    if not path.is_file() or path.stat().st_size == 0:
        return False

    labels = tokenizer.encode(
        text, add_special_tokens=False
    )

    if not (20 <= len(labels) <= 160):
        return False

    if tokenizer.unk_token_id in labels:
        return False

    repeats = sum(
        a == b
        for a, b in zip(labels[:-1], labels[1:])
    )

    frames = count_frames_43(round(duration * 16000))
    return frames - len(labels) - repeats >= 20

def try_select_43(max_duration, allow_digits):
    rng = random.Random(SEED_43)

    groups = defaultdict(list)

    for record in train_records:
        if eligible_43(record, max_duration, allow_digits):
            groups[record["speaker_id"]].append(record)

    for sid in groups:
        groups[sid].sort(key=lambda r: r["id"])
        rng.shuffle(groups[sid])

    # Only unfamiliar speakers can enter validation.
    valid_speakers = [
        sid for sid, records in groups.items()
        if sid not in prior_speakers
        and len(records) >= VAL_PER_SPEAKER
    ]

    valid_speakers.sort()
    rng.shuffle(valid_speakers)

    if len(valid_speakers) < VAL_SPEAKERS:
        return None

    heldout_speakers = set(
        valid_speakers[:VAL_SPEAKERS]
    )

    val_ids = [
        record["id"]
        for sid in valid_speakers[:VAL_SPEAKERS]
        for record in groups[sid][:VAL_PER_SPEAKER]
    ]

    # Carryover examples must remain part of training.
    if not all(
        eligible_43(by_id_43[rid], max_duration, allow_digits)
        for rid in carryover_ids
    ):
        return None

    train_ids = list(carryover_ids)
    used_ids = set(train_ids)

    speaker_counts = Counter(
        by_id_43[rid]["speaker_id"]
        for rid in train_ids
    )

    train_speakers = sorted(
        sid for sid in groups
        if sid not in heldout_speakers
    )
    rng.shuffle(train_speakers)

    # Round robin encourages speaker diversity.
    # Limit to three examples from each speaker.
    for _ in range(3):
        for sid in train_speakers:
            if len(train_ids) >= TRAIN_TARGET:
                break
            if speaker_counts[sid] >= 3:
                continue

            for record in groups[sid]:
                rid = record["id"]

                if rid not in used_ids:
                    train_ids.append(rid)
                    used_ids.add(rid)
                    speaker_counts[sid] += 1
                    break

        if len(train_ids) >= TRAIN_TARGET:
            break

    if len(train_ids) != TRAIN_TARGET:
        return None

    return {
        "train_ids": train_ids,
        "val_ids": val_ids,
        "seed": SEED_43,
        "max_duration_seconds": max_duration,
        "digits_allowed": allow_digits,
        "carryover_ids": carryover_ids,
        "source": "train_final_25h_ctc_safe"
    }

# ------------------------------------------------------------
# Restore or construct selection
# ------------------------------------------------------------

if EXP43_SELECTION.exists():
    selection_43 = json.loads(
        EXP43_SELECTION.read_text(encoding="utf-8")
    )
    print("Existing scaled selection restored.")

else:
    selection_43 = None

    for allow_digits in [False, True]:
        for max_duration in [10, 12, 15, 18]:
            selection_43 = try_select_43(
                max_duration, allow_digits
            )

            if selection_43 is not None:
                print(
                    "Selection succeeded:",
                    max_duration, "seconds,",
                    "digits allowed:", allow_digits
                )
                break

        if selection_43 is not None:
            break

    assert selection_43 is not None, (
        "Not enough eligible recordings. "
        "Review the selection constraints."
    )

    temporary = EXP43_DIR / "selection.json.partial"
    temporary.write_text(
        json.dumps(
            selection_43,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )
    temporary.replace(EXP43_SELECTION)

train_ids_43 = selection_43["train_ids"]
val_ids_43 = selection_43["val_ids"]

assert len(train_ids_43) == TRAIN_TARGET
assert len(val_ids_43) == VAL_TARGET
assert len(set(train_ids_43 + val_ids_43)) == 80
assert set(carryover_ids).issubset(train_ids_43)

train_records_43 = [
    by_id_43[rid] for rid in train_ids_43
]
val_records_43 = [
    by_id_43[rid] for rid in val_ids_43
]

train_speakers_43 = {
    r["speaker_id"] for r in train_records_43
}
val_speakers_43 = {
    r["speaker_id"] for r in val_records_43
}

assert not (train_speakers_43 & val_speakers_43)
assert not (prior_speakers & val_speakers_43)
assert len(val_speakers_43) == VAL_SPEAKERS

assert all(
    eligible_43(
        r,
        selection_43["max_duration_seconds"],
        selection_43["digits_allowed"]
    )
    for r in train_records_43 + val_records_43
)

def duration_summary_43(records):
    values = [
        float(r["duration_seconds"])
        for r in records
    ]

    return (
        round(statistics.mean(values), 2),
        round(max(values), 2),
        round(sum(values) / 60, 2)
    )

print("\n" + "=" * 60)
print("CELL 43: SCALED DATASET SUMMARY")
print("=" * 60)

print("Train recordings:", len(train_records_43))
print("Held-out recordings:", len(val_records_43))
print("Training speakers:", len(train_speakers_43))
print("Held-out speakers:", len(val_speakers_43))
print("Speaker overlap:", 0)
print("Old training examples retained:", len(carryover_ids))
print("Previous experiment speakers in validation:", 0)

print("\nTrain (mean sec, max sec, total min):",
      duration_summary_43(train_records_43))
print("Held-out (mean sec, max sec, total min):",
      duration_summary_43(val_records_43))

print("\nFIRST 5 TRAINING RECORDINGS")
for record in train_records_43[:5]:
    print(
        record["id"],
        round(float(record["duration_seconds"]), 2),
        record["transcription"][:100]
    )

print("\nFIRST 5 HELD-OUT RECORDINGS")
for record in val_records_43[:5]:
    print(
        record["id"],
        round(float(record["duration_seconds"]), 2),
        record["transcription"][:100]
    )

# ------------------------------------------------------------
# Manual audio/transcript inspection
# ------------------------------------------------------------

print("\nAUDIO QUALITY SPOT CHECK")

for label, record in [
    ("NEW TRAINING", train_records_43[8]),
    ("HELD-OUT 1", val_records_43[0]),
    ("HELD-OUT 2", val_records_43[1])
]:
    print("\n", label)
    print("ID:", record["id"])
    print("Transcript:", record["transcription"])

    display(Audio(
        filename=str(
            Path(AUDIO_DIR) / record["audio_file"]
        )
    ))

print("\nDATASET VALIDATION PASSED")
print("Selection saved:", EXP43_SELECTION)
print("No model weights or checkpoints modified.")


## Original source cell 48

Position 49 in the original notebook; original code comments may use different numbering.


In [ ]:

# ============================================================
# CELL 44 - SCALED TRAINING PREFLIGHT
#
# Restore Branch B from Cell 42D, update 300.
# Verify gradients on the longest training recording.
# Evaluate 16 new held-out recordings.
#
# No optimizer updates or checkpoint overwrites.
# ============================================================

import json
import gc
import statistics
from pathlib import Path

import torch
from jiwer import cer, wer
from transformers import Wav2Vec2ForCTC

DEVICE_44 = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

SOURCE_CHECKPOINT = (
    DATA_DIR
    / "warm_head_encoder_comparison_v1"
    / "last2_fp32_latest.pt"
)

SELECTION_44 = (
    DATA_DIR
    / "scaled_amharic_64_16_v1"
    / "selection.json"
)

PREFLIGHT_PATH = (
    DATA_DIR
    / "scaled_amharic_64_16_v1"
    / "fp32_preflight.json"
)

assert SOURCE_CHECKPOINT.is_file()
assert SELECTION_44.is_file()

selection_44 = json.loads(
    SELECTION_44.read_text(encoding="utf-8")
)

train_ids_44 = selection_44["train_ids"]
val_ids_44 = selection_44["val_ids"]

assert len(train_ids_44) == 64
assert len(val_ids_44) == 16

record_by_id_44 = {
    record["id"]: record
    for record in train_records
}

dataset_index_44 = {
    record["id"]: index
    for index, record in enumerate(train_dataset.records)
}

assert all(
    rid in dataset_index_44
    for rid in train_ids_44 + val_ids_44
)

train_speakers_44 = {
    record_by_id_44[rid]["speaker_id"]
    for rid in train_ids_44
}

val_speakers_44 = {
    record_by_id_44[rid]["speaker_id"]
    for rid in val_ids_44
}

assert not (train_speakers_44 & val_speakers_44)

# ------------------------------------------------------------
# 1. Load the successful Branch B checkpoint
# ------------------------------------------------------------

print("LOADING BRANCH B")
print("=" * 55)

saved_44 = torch.load(
    SOURCE_CHECKPOINT,
    map_location="cpu",
    weights_only=True
)

assert saved_44["precision"] == "fp32"
assert int(saved_44["step"]) == 300
assert len(saved_44["upper_layers"]) == 2

model_44 = Wav2Vec2ForCTC.from_pretrained(
    str(BASELINE_DIR)
).to(DEVICE_44).float()

model_44.lm_head.load_state_dict(
    saved_44["head"],
    strict=True
)

upper_layers_44 = list(
    model_44.wav2vec2.encoder.layers[-2:]
)

for layer, weights in zip(
    upper_layers_44,
    saved_44["upper_layers"]
):
    layer.load_state_dict(weights, strict=True)

# Freeze all parameters.
model_44.requires_grad_(False)

# Enable the same components as Branch B.
for layer in upper_layers_44:
    layer.requires_grad_(True)

model_44.lm_head.requires_grad_(True)

# Keep dropout and SpecAugment disabled
# during this controlled experiment.
model_44.eval()

trainable_44 = [
    parameter
    for parameter in model_44.parameters()
    if parameter.requires_grad
]

assert sum(
    p.numel() for p in trainable_44
) == 25521473

print("Checkpoint step: 300")
print("Precision: FP32")
print("Trainable parameters:", sum(
    p.numel() for p in trainable_44
))

# ------------------------------------------------------------
# 2. Helper: Retrieve one recording
# ------------------------------------------------------------

def get_batch_44(record_id):

    example = train_dataset[
        dataset_index_44[record_id]
    ]

    batch = ctc_collator([example])

    return {
        key: value.to(DEVICE_44)
        for key, value in batch.items()
    }

def decode_44(logits):

    predicted_ids = logits.argmax(dim=-1)

    prediction = processor.batch_decode(
        predicted_ids.cpu()
    )[0]

    return normalize_amharic_ctc(
        prediction
    )

# ------------------------------------------------------------
# 3. Evaluate the new validation set
# ------------------------------------------------------------

print("\nNEW VALIDATION BASELINE")
print("=" * 55)

references_44 = []
predictions_44 = []
validation_rows_44 = []

model_44.eval()

with torch.no_grad():

    for index, rid in enumerate(val_ids_44, 1):

        batch = get_batch_44(rid)

        logits = model_44(**batch).logits
        prediction = decode_44(logits)

        reference = normalize_amharic_ctc(
            record_by_id_44[rid]["transcription"]
        )

        references_44.append(reference)
        predictions_44.append(prediction)

        validation_rows_44.append({
            "id": rid,
            "reference": reference,
            "prediction": prediction,
            "cer": float(cer(reference, prediction)),
            "wer": float(wer(reference, prediction)),
            "duration": float(
                record_by_id_44[rid]["duration_seconds"]
            )
        })

        if index <= 3:
            print(f"\nValidation recording {index}")
            print("ID:", rid)
            print("Reference:", reference[:150])
            print(
                "Prediction:",
                prediction[:150] or "[EMPTY]"
            )

        del batch, logits

baseline_cer_44 = float(
    cer(references_44, predictions_44)
)

baseline_wer_44 = float(
    wer(references_44, predictions_44)
)

print("\nValidation CER:",
      f"{baseline_cer_44 * 100:.2f}%")

print("Validation WER:",
      f"{baseline_wer_44 * 100:.2f}%")

print("Nonempty predictions:", sum(
    bool(p.strip()) for p in predictions_44
), "/16")

# ------------------------------------------------------------
# 4. Test gradients on the LONGEST training recording
# ------------------------------------------------------------

longest_id_44 = max(
    train_ids_44,
    key=lambda rid: float(
        record_by_id_44[rid]["duration_seconds"]
    )
)

longest_duration_44 = float(
    record_by_id_44[
        longest_id_44
    ]["duration_seconds"]
)

print("\nLONG-RECORDING FP32 GRADIENT TEST")
print("=" * 55)

print("Recording:", longest_id_44)
print("Duration:", longest_duration_44, "seconds")

test_batch_44 = get_batch_44(longest_id_44)

# Confirm actual CTC alignment feasibility.
target_ids_44 = test_batch_44["labels"][
    test_batch_44["labels"] >= 0
]

actual_frames_44 = int(
    model_44._get_feat_extract_output_lengths(
        torch.tensor([
            test_batch_44["input_values"].shape[1]
        ])
    )[0].item()
)

repeats_44 = int(
    (target_ids_44[1:] == target_ids_44[:-1])
    .sum().item()
)

required_frames_44 = (
    target_ids_44.numel() + repeats_44
)

print("Available CTC frames:", actual_frames_44)
print("Required CTC frames:", required_frames_44)

assert actual_frames_44 >= required_frames_44

# Free existing gradients from previous work.
model_44.zero_grad(set_to_none=True)

if DEVICE_44.type == "cuda":
    torch.cuda.reset_peak_memory_stats()

# No autocast. No gradient scaler.
with torch.autocast(
    device_type=DEVICE_44.type,
    enabled=False
):
    test_loss_44 = model_44(**test_batch_44).loss

print("\nLoss:", float(test_loss_44.item()))
print("Loss dtype:", test_loss_44.dtype)
print("Requires grad:", test_loss_44.requires_grad)

assert torch.isfinite(test_loss_44).item()
assert test_loss_44.requires_grad

test_loss_44.backward()

# ------------------------------------------------------------
# 5. Verify gradients before any training
# ------------------------------------------------------------

invalid_44 = []
missing_44 = []
valid_44 = 0

for name, parameter in model_44.named_parameters():

    if not parameter.requires_grad:
        continue

    if parameter.grad is None:
        missing_44.append(name)
        continue

    if not torch.isfinite(parameter.grad).all().item():
        invalid_44.append(name)
    else:
        valid_44 += 1

print("\nGRADIENT RESULTS")
print("Valid parameter gradients:", valid_44)
print("Missing parameter gradients:", len(missing_44))
print("Invalid parameter gradients:", len(invalid_44))

assert not invalid_44, (
    f"Invalid gradients: {invalid_44[:5]}"
)

assert not missing_44, (
    f"Missing gradients: {missing_44[:5]}"
)

gradient_norm_44 = torch.nn.utils.clip_grad_norm_(
    trainable_44,
    max_norm=1.0,
    error_if_nonfinite=True
)

print(
    "Gradient norm before clipping:",
    round(float(gradient_norm_44), 4)
)

peak_gpu_gb_44 = None

if DEVICE_44.type == "cuda":
    peak_gpu_gb_44 = (
        torch.cuda.max_memory_allocated()
        / 1024**3
    )

    print(
        "Peak allocated GPU memory:",
        round(peak_gpu_gb_44, 2),
        "GB"
    )

# Remove gradients from the diagnostic.
model_44.zero_grad(set_to_none=True)

del test_batch_44, test_loss_44
gc.collect()

if DEVICE_44.type == "cuda":
    torch.cuda.empty_cache()

# ------------------------------------------------------------
# 6. Save baseline measurements
# ------------------------------------------------------------

preflight_report_44 = {
    "source_checkpoint": str(SOURCE_CHECKPOINT),
    "source_step": 300,
    "training_examples": len(train_ids_44),
    "validation_examples": len(val_ids_44),
    "validation_cer": baseline_cer_44,
    "validation_wer": baseline_wer_44,
    "validation_predictions": validation_rows_44,
    "longest_train_recording": longest_id_44,
    "longest_train_duration": longest_duration_44,
    "gradient_norm": float(gradient_norm_44),
    "valid_gradient_tensors": valid_44,
    "invalid_gradient_tensors": len(invalid_44),
    "peak_allocated_gpu_gb": peak_gpu_gb_44
}

PREFLIGHT_PATH.write_text(
    json.dumps(
        preflight_report_44,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print("\n" + "=" * 55)
print("SCALED TRAINING PREFLIGHT PASSED")
print("=" * 55)

print("Validation CER:",
      f"{baseline_cer_44 * 100:.2f}%")
print("Validation WER:",
      f"{baseline_wer_44 * 100:.2f}%")
print("Full-precision gradients: VALID")
print("Optimizer updates: 0")
print("Baseline saved:", PREFLIGHT_PATH)
print("Existing checkpoints: UNCHANGED")


## Original source cell 49

Position 50 in the original notebook; original code comments may use different numbering.


In [ ]:

# ============================================================
# CELL 45 - SCALED FP32 TRAINING PILOT
#
# 64 training recordings, 16 held-out recordings
# Start from Branch B at update 300
# Train last 2 Transformer layers + CTC head
#
# 128 updates (2 passes through 64 recordings)
# Evaluate and checkpoint every 32 updates
# ============================================================

import json
import random
import hashlib
import statistics
from pathlib import Path

import torch
from jiwer import cer, wer

PILOT_STEPS = 128
REPORT_INTERVAL = 32

HEAD_LR_45 = 1e-4
ENCODER_LR_45 = 2e-6
CLIP_NORM_45 = 1.0

PILOT_DIR = (
    DATA_DIR
    / "scaled_amharic_64_16_v1"
    / "stage1_fp32"
)
PILOT_DIR.mkdir(parents=True, exist_ok=True)

STATE_45 = PILOT_DIR / "latest.pt"
HISTORY_45 = PILOT_DIR / "history.json"

required_names = [
    "model_44", "saved_44",
    "train_ids_44", "val_ids_44",
    "get_batch_44", "record_by_id_44",
    "decode_44"
]

missing = [
    name for name in required_names
    if name not in globals()
]
assert not missing, f"Missing Cell 44 state: {missing}"

assert len(train_ids_44) == 64
assert len(val_ids_44) == 16
assert int(saved_44["step"]) == 300

# ------------------------------------------------------------
# 1. Reconfigure the existing model
# ------------------------------------------------------------

model_44 = model_44.float()
model_44.requires_grad_(False)

upper_45 = list(
    model_44.wav2vec2.encoder.layers[-2:]
)

for layer in upper_45:
    layer.requires_grad_(True)

model_44.lm_head.requires_grad_(True)
model_44.eval()

encoder_params_45 = [
    p for layer in upper_45
    for p in layer.parameters()
    if p.requires_grad
]

head_params_45 = list(
    model_44.lm_head.parameters()
)

trainable_45 = encoder_params_45 + head_params_45

assert sum(p.numel() for p in trainable_45) == 25521473

optimizer_45 = torch.optim.AdamW([
    {
        "params": encoder_params_45,
        "lr": ENCODER_LR_45,
        "weight_decay": 0.01
    },
    {
        "params": head_params_45,
        "lr": HEAD_LR_45,
        "weight_decay": 0.0
    }
])

# A new optimizer starts here. We retain trained
# weights but reset AdamW's moment estimates.

config_45 = {
    "source": str(SOURCE_CHECKPOINT),
    "source_step": 300,
    "train_ids": train_ids_44,
    "val_ids": val_ids_44,
    "references": {
        rid: record_by_id_44[rid]["transcription"]
        for rid in train_ids_44 + val_ids_44
    },
    "head_lr": HEAD_LR_45,
    "encoder_lr": ENCODER_LR_45,
    "clip_norm": CLIP_NORM_45,
    "seed": 451
}

signature_45 = hashlib.sha256(
    json.dumps(
        config_45,
        sort_keys=True,
        ensure_ascii=False
    ).encode("utf-8")
).hexdigest()

# ------------------------------------------------------------
# 2. Restore saved pilot progress or warm-start weights
# ------------------------------------------------------------

start_45 = 0
history_45 = []

if STATE_45.exists():

    saved = torch.load(
        STATE_45,
        map_location="cpu",
        weights_only=True
    )

    assert saved["signature"] == signature_45

    model_44.lm_head.load_state_dict(saved["head"])

    for layer, weights in zip(
        upper_45, saved["upper_layers"]
    ):
        layer.load_state_dict(weights)

    optimizer_45.load_state_dict(saved["optimizer"])
    start_45 = int(saved["step"])

    if HISTORY_45.exists():
        history_45 = json.loads(
            HISTORY_45.read_text(encoding="utf-8")
        )

    print("Resuming pilot from update:", start_45)

else:

    # Restore exact Branch B update-300 weights.
    model_44.lm_head.load_state_dict(
        saved_44["head"]
    )

    for layer, weights in zip(
        upper_45, saved_44["upper_layers"]
    ):
        layer.load_state_dict(weights)

    print("Starting from verified Branch B update 300.")

assert 0 <= start_45 <= PILOT_STEPS

# ------------------------------------------------------------
# 3. Evaluation
# ------------------------------------------------------------

# Eight previously trained examples plus eight newly
# selected training examples form a fixed training probe.
train_probe_45 = (
    train_ids_44[:8] + train_ids_44[8:16]
)

def evaluate_45(record_ids):

    model_44.eval()

    refs = []
    preds = []

    with torch.no_grad():

        for rid in record_ids:

            batch = get_batch_44(rid)

            output = model_44(
                input_values=batch["input_values"],
                attention_mask=batch["attention_mask"]
            )

            prediction = decode_44(output.logits)

            reference = normalize_amharic_ctc(
                record_by_id_44[rid]["transcription"]
            )

            refs.append(reference)
            preds.append(prediction)

            del output, batch

    return {
        "cer": float(cer(refs, preds)),
        "wer": float(wer(refs, preds)),
        "nonempty": sum(bool(p.strip()) for p in preds),
        "predictions": preds
    }

def report_45(step):

    train_result = evaluate_45(train_probe_45)
    val_result = evaluate_45(val_ids_44)

    result = {
        "step": step,
        "train_probe_cer": train_result["cer"],
        "val_cer": val_result["cer"],
        "val_wer": val_result["wer"],
        "val_nonempty": val_result["nonempty"],
        "train_prediction": train_result["predictions"][0],
        "val_prediction": val_result["predictions"][0]
    }

    print(f"\nUPDATE {step}/{PILOT_STEPS}")
    print("-" * 50)

    print(
        "Training probe CER (16 recordings):",
        f"{result['train_probe_cer'] * 100:.2f}%"
    )

    print(
        "Held-out CER (16 recordings):",
        f"{result['val_cer'] * 100:.2f}%"
    )

    print(
        "Held-out WER:",
        f"{result['val_wer'] * 100:.2f}%"
    )

    print(
        "Held-out nonempty predictions:",
        f"{result['val_nonempty']}/16"
    )

    print(
        "Held-out example:",
        result["val_prediction"][:120] or "[EMPTY]"
    )

    return result

def save_45(step):

    checkpoint = {
        "signature": signature_45,
        "step": step,
        "precision": "fp32",
        "head": model_44.lm_head.state_dict(),
        "upper_layers": [
            layer.state_dict()
            for layer in upper_45
        ],
        "optimizer": optimizer_45.state_dict()
    }

    temp_state = PILOT_DIR / "latest.pt.partial"
    torch.save(checkpoint, temp_state)
    temp_state.replace(STATE_45)

    temp_history = PILOT_DIR / "history.json.partial"

    temp_history.write_text(
        json.dumps(
            history_45,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )

    temp_history.replace(HISTORY_45)

# ------------------------------------------------------------
# 4. Establish baseline
# ------------------------------------------------------------

print("\nSCALED FP32 PILOT")
print("=" * 55)
print("Training recordings: 64")
print("Held-out recordings: 16")
print("Trainable parameters: 25,521,473")
print("Head LR:", HEAD_LR_45)
print("Encoder LR:", ENCODER_LR_45)
print("Gradient clip:", CLIP_NORM_45)
print("FP16 autocast: DISABLED")

if start_45 == 0 and not history_45:

    baseline_45 = report_45(0)

    preflight = json.loads(
        PREFLIGHT_PATH.read_text(encoding="utf-8")
    )

    # Confirm the clean initial validation model.
    assert abs(
        baseline_45["val_cer"]
        - preflight["validation_cer"]
    ) < 0.01, "Baseline mismatch; stop before training."

    history_45.append(baseline_45)

# ------------------------------------------------------------
# 5. Deterministic speaker-diverse data order
# ------------------------------------------------------------

def record_for_step_45(step):

    zero_index = step - 1
    epoch = zero_index // len(train_ids_44)
    position = zero_index % len(train_ids_44)

    order = list(train_ids_44)

    rng = random.Random(451 + epoch)
    rng.shuffle(order)

    return order[position]

# ------------------------------------------------------------
# 6. Run 128 optimizer updates
# ------------------------------------------------------------

recent_norms = []
recent_losses = []
completed_45 = start_45

for step in range(start_45 + 1, PILOT_STEPS + 1):

    rid = record_for_step_45(step)

    model_44.eval()
    optimizer_45.zero_grad(set_to_none=True)

    batch = get_batch_44(rid)

    # Full precision, including CTC calculation.
    with torch.autocast(
        device_type=DEVICE_44.type,
        enabled=False
    ):
        loss = model_44(**batch).loss

    if not torch.isfinite(loss).item():
        raise RuntimeError(
            f"Non-finite loss at update {step}, ID {rid}"
        )

    loss.backward()

    grad_norm = torch.nn.utils.clip_grad_norm_(
        trainable_45,
        max_norm=CLIP_NORM_45,
        error_if_nonfinite=True
    )

    raw_norm = float(grad_norm)

    # Stop on extreme unexpected gradients.
    if raw_norm > 5000:
        optimizer_45.zero_grad(set_to_none=True)
        raise RuntimeError(
            f"Extreme gradient norm {raw_norm:.1f} "
            f"at update {step}, recording {rid}"
        )

    optimizer_45.step()

    recent_norms.append(raw_norm)
    recent_losses.append(float(loss.item()))

    completed_45 = step

    del batch, loss

    if step % REPORT_INTERVAL == 0:

        result = report_45(step)

        result["mean_train_step_loss"] = (
            statistics.mean(recent_losses)
        )
        result["median_gradient_norm"] = (
            statistics.median(recent_norms)
        )
        result["max_gradient_norm"] = max(
            recent_norms
        )
        result["fraction_gradients_clipped"] = (
            sum(n > CLIP_NORM_45 for n in recent_norms)
            / len(recent_norms)
        )

        print(
            "Mean optimizer-step loss:",
            round(result["mean_train_step_loss"], 4)
        )

        print(
            "Median gradient norm:",
            round(result["median_gradient_norm"], 2)
        )

        print(
            "Maximum gradient norm:",
            round(result["max_gradient_norm"], 2)
        )

        print(
            "Fraction of steps clipped:",
            f"{result['fraction_gradients_clipped'] * 100:.1f}%"
        )

        history_45.append(result)
        save_45(step)

        recent_norms.clear()
        recent_losses.clear()

# ------------------------------------------------------------
# 7. Summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("SCALED FP32 PILOT SUMMARY")
print("=" * 65)

print(
    f"{'Update':<9}"
    f"{'Train probe CER':<19}"
    f"{'Held-out CER':<17}"
    f"{'Held-out WER':<17}"
)

for row in history_45:

    print(
        f"{row['step']:<9}"
        f"{row['train_probe_cer']*100:<19.2f}"
        f"{row['val_cer']*100:<17.2f}"
        f"{row['val_wer']*100:<17.2f}"
    )

print("\nUpdates completed:", completed_45)
print("Checkpoint:", STATE_45)
print("History:", HISTORY_45)
print("Earlier experiments and checkpoints unchanged.")


## Original source cell 50

Position 51 in the original notebook; original code comments may use different numbering.


In [ ]:

# ============================================================
# CELL 46 - SCALED ASR FAILURE-MODE AUDIT
#
# Evaluate 64 training + 16 held-out recordings.
# Measure CER, WER, CTC loss, blank behavior,
# output length, and transcription density.
#
# No training. No checkpoint modifications.
# ============================================================

import json
import statistics
from collections import defaultdict
from pathlib import Path

import torch
from transformers import Wav2Vec2ForCTC
from jiwer import cer, wer

AUDIT_DIR = (
    DATA_DIR / "scaled_amharic_64_16_v1"
)
AUDIT_CHECKPOINT = (
    AUDIT_DIR / "stage1_fp32" / "latest.pt"
)
AUDIT_SELECTION = AUDIT_DIR / "selection.json"
AUDIT_BASELINE = AUDIT_DIR / "fp32_preflight.json"
AUDIT_OUTPUT = AUDIT_DIR / "failure_mode_audit.json"

assert AUDIT_CHECKPOINT.is_file()
assert AUDIT_SELECTION.is_file()
assert AUDIT_BASELINE.is_file()

device_46 = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

selection_46 = json.loads(
    AUDIT_SELECTION.read_text(encoding="utf-8")
)
baseline_46 = json.loads(
    AUDIT_BASELINE.read_text(encoding="utf-8")
)
saved_46 = torch.load(
    AUDIT_CHECKPOINT,
    map_location="cpu",
    weights_only=True
)

assert saved_46["step"] == 128
assert saved_46["precision"] == "fp32"

train_ids_46 = selection_46["train_ids"]
val_ids_46 = selection_46["val_ids"]
old_train_ids_46 = set(
    selection_46["carryover_ids"]
)

assert len(train_ids_46) == 64
assert len(val_ids_46) == 16
assert len(old_train_ids_46) == 8

record_lookup_46 = {
    r["id"]: r for r in train_records
}
dataset_lookup_46 = {
    r["id"]: i
    for i, r in enumerate(train_dataset.records)
}

# ------------------------------------------------------------
# 1. Load saved 128-update model independently
# ------------------------------------------------------------

print("LOADING SAVED PILOT MODEL")
print("=" * 58)

model_46 = Wav2Vec2ForCTC.from_pretrained(
    str(BASELINE_DIR)
).to(device_46).float()

model_46.lm_head.load_state_dict(
    saved_46["head"], strict=True
)

for layer, weights in zip(
    model_46.wav2vec2.encoder.layers[-2:],
    saved_46["upper_layers"]
):
    layer.load_state_dict(weights, strict=True)

model_46.eval()
model_46.requires_grad_(False)

blank_46 = tokenizer.pad_token_id

print("Checkpoint: update", saved_46["step"])
print("Recordings to evaluate: 80")
print("Model in evaluation mode:", not model_46.training)

# ------------------------------------------------------------
# 2. Evaluate all recordings
# ------------------------------------------------------------

rows_46 = []

def evaluate_record_46(rid, group):

    record = record_lookup_46[rid]
    example = train_dataset[
        dataset_lookup_46[rid]
    ]
    batch = ctc_collator([example])

    batch = {
        k: v.to(device_46)
        for k, v in batch.items()
    }

    actual_seconds = (
        int(batch["attention_mask"].sum().item())
        / 16000
    )

    metadata_seconds = float(
        record["duration_seconds"]
    )

    with torch.no_grad():
        output = model_46(**batch)

        logits = output.logits.float()
        frame_ids = logits.argmax(dim=-1)

        blank_fraction = (
            (frame_ids == blank_46)
            .float()
            .mean()
            .item()
        )

        blank_probability = (
            logits.softmax(dim=-1)[
                :, :, blank_46
            ]
            .mean()
            .item()
        )

        prediction = processor.batch_decode(
            frame_ids.cpu()
        )[0]

    prediction = normalize_amharic_ctc(
        prediction
    )

    reference = normalize_amharic_ctc(
        record["transcription"]
    )

    ref_chars = len(reference.replace(" ", ""))
    pred_chars = len(prediction.replace(" ", ""))

    return {
        "id": rid,
        "group": group,
        "speaker": record["speaker_id"],
        "metadata_seconds": metadata_seconds,
        "audio_seconds": actual_seconds,
        "reference": reference,
        "prediction": prediction,
        "reference_chars": ref_chars,
        "predicted_chars": pred_chars,
        "chars_per_second":
            ref_chars / max(actual_seconds, 0.01),
        "prediction_length_ratio":
            pred_chars / max(ref_chars, 1),
        "cer": float(cer(reference, prediction)),
        "wer": float(wer(reference, prediction)),
        "ctc_loss": float(output.loss.item()),
        "blank_argmax_fraction": blank_fraction,
        "mean_blank_probability": blank_probability
    }

groups_46 = {
    "Previous 8 training": [
        rid for rid in train_ids_46
        if rid in old_train_ids_46
    ],
    "New 56 training": [
        rid for rid in train_ids_46
        if rid not in old_train_ids_46
    ],
    "Held-out 16": list(val_ids_46)
}

for group_name, ids in groups_46.items():
    print(f"\nEvaluating {group_name}...")

    for i, rid in enumerate(ids, 1):
        rows_46.append(
            evaluate_record_46(rid, group_name)
        )

        if i % 10 == 0 or i == len(ids):
            print(f"  {i}/{len(ids)}")

# ------------------------------------------------------------
# 3. Aggregate recognition metrics by group
# ------------------------------------------------------------

def summarize_group_46(group_name):

    group = [
        r for r in rows_46
        if r["group"] == group_name
    ]

    return {
        "count": len(group),
        "cer": cer(
            [r["reference"] for r in group],
            [r["prediction"] for r in group]
        ),
        "wer": wer(
            [r["reference"] for r in group],
            [r["prediction"] for r in group]
        ),
        "mean_ctc_loss": statistics.mean(
            r["ctc_loss"] for r in group
        ),
        "blank_frames": statistics.mean(
            r["blank_argmax_fraction"]
            for r in group
        ),
        "prediction_length_ratio": (
            sum(r["predicted_chars"] for r in group)
            / max(
                sum(r["reference_chars"] for r in group),
                1
            )
        )
    }

summary_46 = {
    name: summarize_group_46(name)
    for name in groups_46
}

print("\n" + "=" * 70)
print("RECOGNITION FAILURE-MODE SUMMARY")
print("=" * 70)

print(
    f"{'Group':<21}"
    f"{'CER':>8}"
    f"{'WER':>8}"
    f"{'CTC loss':>11}"
    f"{'Blank %':>10}"
    f"{'Length':>9}"
)

for name, s in summary_46.items():
    print(
        f"{name:<21}"
        f"{100*s['cer']:>7.1f}%"
        f"{100*s['wer']:>7.1f}%"
        f"{s['mean_ctc_loss']:>11.2f}"
        f"{100*s['blank_frames']:>9.1f}%"
        f"{s['prediction_length_ratio']:>9.2f}"
    )

print("\nLength = predicted characters / reference characters")
print("Blank % = fraction of frames decoded as CTC blank")

# ------------------------------------------------------------
# 4. Compare each validation recording before vs after
# ------------------------------------------------------------

baseline_by_id_46 = {
    r["id"]: r
    for r in baseline_46["validation_predictions"]
}

changes_46 = []

for row in rows_46:

    if row["group"] != "Held-out 16":
        continue

    before = baseline_by_id_46[row["id"]]["cer"]
    after = row["cer"]

    changes_46.append({
        "id": row["id"],
        "before": before,
        "after": after,
        "change": after - before
    })

changes_46.sort(
    key=lambda row: row["change"]
)

print("\n" + "=" * 65)
print("PER-RECORDING VALIDATION CHANGE")
print("=" * 65)

print("Most improved (negative CER change is better):")

for row in changes_46[:5]:
    print(
        row["id"],
        f"{row['before']*100:.1f}% ->",
        f"{row['after']*100:.1f}%",
        f"(change {row['change']*100:+.1f} pp)"
    )

print("\nMost degraded:")

for row in changes_46[-5:]:
    print(
        row["id"],
        f"{row['before']*100:.1f}% ->",
        f"{row['after']*100:.1f}%",
        f"(change {row['change']*100:+.1f} pp)"
    )

# ------------------------------------------------------------
# 5. Identify unusual transcript/audio ratios
# ------------------------------------------------------------

by_density_46 = sorted(
    rows_46,
    key=lambda row: row["chars_per_second"]
)

duration_mismatches_46 = [
    row for row in rows_46
    if abs(
        row["audio_seconds"]
        - row["metadata_seconds"]
    ) > 0.5
]

print("\n" + "=" * 65)
print("TRANSCRIPT QUALITY AUDIT")
print("=" * 65)

print("Recording duration mismatches (>0.5s):",
      len(duration_mismatches_46))

for row in duration_mismatches_46[:8]:
    print(
        row["id"],
        "metadata:", round(row["metadata_seconds"], 2),
        "audio:", round(row["audio_seconds"], 2)
    )

print("\nLowest transcript characters per second:")
print("(Candidates for manual review, not proven errors)")

for row in by_density_46[:10]:
    print(
        row["id"],
        f"| {row['audio_seconds']:.1f}s",
        f"| {row['chars_per_second']:.2f} chars/s",
        f"| loss {row['ctc_loss']:.2f}",
        "|", row["group"]
    )

# ------------------------------------------------------------
# 6. Save complete audit
# ------------------------------------------------------------

audit_46 = {
    "checkpoint_step": int(saved_46["step"]),
    "summary": summary_46,
    "validation_changes": changes_46,
    "duration_mismatches": duration_mismatches_46,
    "rows": rows_46
}

AUDIT_OUTPUT.write_text(
    json.dumps(
        audit_46,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print("\n" + "=" * 65)
print("AUDIT COMPLETE")
print("=" * 65)
print("Saved:", AUDIT_OUTPUT)
print("Optimizer updates: 0")
print("Checkpoints modified: 0")


## Original source cell 51

Position 52 in the original notebook; original code comments may use different numbering.


In [ ]:

# ============================================================
# CELL 47 - TARGETED AUDIO / TRANSCRIPT QUALITY REVIEW
#
# Listen to recordings most likely to reveal
# transcription, clipping, or silence problems.
#
# No model loading or training.
# Existing selections/checkpoints remain unchanged.
# ============================================================

import json
import statistics
from pathlib import Path
from IPython.display import Audio, display

AUDIT_FILE_47 = (
    DATA_DIR / "scaled_amharic_64_16_v1"
    / "failure_mode_audit.json"
)

assert AUDIT_FILE_47.exists()

audit_47 = json.loads(
    AUDIT_FILE_47.read_text(encoding="utf-8")
)

rows_47 = audit_47["rows"]
by_id_47 = {r["id"]: r for r in rows_47}

record_by_id_47 = {
    r["id"]: r for r in train_records
}

old_47 = [
    r for r in rows_47
    if r["group"] == "Previous 8 training"
]

new_47 = [
    r for r in rows_47
    if r["group"] == "New 56 training"
]

val_47 = [
    r for r in rows_47
    if r["group"] == "Held-out 16"
]

print("REFERENCE CHARACTER DENSITY")
print("=" * 55)

for name, group in [
    ("Previously learned", old_47),
    ("New training", new_47),
    ("Held-out", val_47)
]:
    densities = [
        r["chars_per_second"]
        for r in group
    ]

    print(
        f"{name:<20}",
        f"median={statistics.median(densities):.2f}",
        f"min={min(densities):.2f}",
        f"max={max(densities):.2f}"
    )

# ------------------------------------------------------------
# 1. Select diagnostic examples
# ------------------------------------------------------------

review_47 = []
seen_47 = set()

def add_review_47(reason, row):
    if row["id"] not in seen_47:
        review_47.append((reason, row))
        seen_47.add(row["id"])

# Known positive control
anchor_id_47 = "6Q0fEziAqyQVIz4J8aws"

add_review_47(
    "POSITIVE CONTROL: previously learned recording",
    by_id_47[anchor_id_47]
)

# Lowest transcript density among new training examples
add_review_47(
    "TRAIN: unusually sparse transcript",
    min(
        new_47,
        key=lambda r: r["chars_per_second"]
    )
)

# Highest CTC loss not already selected
for row in sorted(
    new_47,
    key=lambda r: r["ctc_loss"],
    reverse=True
):
    if row["id"] not in seen_47:
        add_review_47(
            "TRAIN: highest remaining CTC loss",
            row
        )
        break

# Lowest transcript density in validation
add_review_47(
    "HELD-OUT: unusually sparse transcript",
    min(
        val_47,
        key=lambda r: r["chars_per_second"]
    )
)

# Largest deterioration since the validation baseline
changes_47 = audit_47["validation_changes"]

most_degraded_47 = max(
    changes_47,
    key=lambda r: r["change"]
)

add_review_47(
    "HELD-OUT: largest CER deterioration",
    by_id_47[most_degraded_47["id"]]
)

assert len(review_47) == 5

# ------------------------------------------------------------
# 2. Display audio and transcript side by side
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("TARGETED AUDIO REVIEW")
print("=" * 65)

review_log_47 = []

for index, (reason, row) in enumerate(review_47, 1):

    record = record_by_id_47[row["id"]]

    audio_path = (
        Path(AUDIO_DIR) / record["audio_file"]
    )

    assert audio_path.exists(), audio_path

    print("\n" + "=" * 65)
    print(f"RECORDING {index}/5")
    print("=" * 65)

    print("Reason:", reason)
    print("ID:", row["id"])
    print("Group:", row["group"])

    print(
        "Audio duration:",
        f"{row['audio_seconds']:.2f}s"
    )

    print(
        "Reference density:",
        f"{row['chars_per_second']:.2f} chars/sec"
    )

    print(
        "CTC loss:",
        f"{row['ctc_loss']:.3f}"
    )

    print(
        "CER:",
        f"{row['cer'] * 100:.2f}%"
    )

    print(
        "Predicted/reference length:",
        f"{row['prediction_length_ratio']:.2f}"
    )

    print("\nREFERENCE TRANSCRIPT:")
    print(row["reference"])

    print("\nMODEL PREDICTION:")
    print(row["prediction"] or "[EMPTY]")

    print("\nLISTEN TO AUDIO:")
    display(Audio(filename=str(audio_path)))

    review_log_47.append({
        "id": row["id"],
        "reason": reason,
        "group": row["group"],
        "duration_seconds": row["audio_seconds"],
        "reference": row["reference"],
        "cer": row["cer"],
        "ctc_loss": row["ctc_loss"],
        "review_status": "not_yet_reviewed"
    })

# ------------------------------------------------------------
# 3. Save review queue separately
# ------------------------------------------------------------

REVIEW_FILE_47 = (
    DATA_DIR / "scaled_amharic_64_16_v1"
    / "manual_review_queue.json"
)

if not REVIEW_FILE_47.exists():
    REVIEW_FILE_47.write_text(
        json.dumps(
            review_log_47,
            ensure_ascii=False,
            indent=2
        ),
        encoding="utf-8"
    )

print("\n" + "=" * 65)
print("REVIEW QUEUE CREATED")
print("=" * 65)

print("Recordings to listen to:", len(review_47))
print("Review queue:", REVIEW_FILE_47)
print("Training updates: 0")
print("Checkpoints modified: 0")


## Original source cell 52

Position 53 in the original notebook; original code comments may use different numbering.


In [ ]:

# ============================================================
# CELL 48 - APPLY MANUAL QUALITY REVIEW
#
# Record five human annotations from Cell 47.
# Create a provisional filtered dataset.
# Select five additional recordings for review.
#
# No training and no checkpoint modifications.
# ============================================================

import json
from pathlib import Path
from IPython.display import Audio, display

QA_DIR = DATA_DIR / "scaled_amharic_64_16_v1"

REVIEW_QUEUE = QA_DIR / "manual_review_queue.json"
AUDIT_PATH = QA_DIR / "failure_mode_audit.json"
SELECTION_PATH = QA_DIR / "selection.json"

LABELS_PATH = QA_DIR / "manual_review_results_round1.json"
FILTERED_PATH = QA_DIR / "quality_filtered_v1.json"
ROUND2_PATH = QA_DIR / "manual_review_round2.json"

review_queue = json.loads(
    REVIEW_QUEUE.read_text(encoding="utf-8")
)
audit = json.loads(
    AUDIT_PATH.read_text(encoding="utf-8")
)
selection = json.loads(
    SELECTION_PATH.read_text(encoding="utf-8")
)

assert len(review_queue) == 5

audit_rows = {
    row["id"]: row
    for row in audit["rows"]
}

# ------------------------------------------------------------
# 1. Save your review results in the displayed order
# ------------------------------------------------------------

decisions = [
    (
        "quarantine",
        "Partial transcript with additional speech"
    ),
    (
        "quarantine",
        "Silence reported during listening"
    ),
    (
        "verified_match",
        "Spoken content matches transcript"
    ),
    (
        "noisy_challenge",
        "Heavy background noise and repeated words"
    ),
    (
        "verified_match",
        "Spoken content matches transcript"
    )
]

expected_groups = [
    "Previous 8 training",
    "New 56 training",
    "New 56 training",
    "Held-out 16",
    "Held-out 16"
]

annotations = []

for index, (item, decision) in enumerate(
    zip(review_queue, decisions)
):

    status, reason = decision
    rid = item["id"]

    assert rid in audit_rows
    assert audit_rows[rid]["group"] == expected_groups[index]

    annotations.append({
        "id": rid,
        "original_group": expected_groups[index],
        "review_status": status,
        "finding": reason
    })

# Prevent accidental overwriting of different
# human annotations from a previous run.
if LABELS_PATH.exists():
    existing = json.loads(
        LABELS_PATH.read_text(encoding="utf-8")
    )
    assert existing == annotations, (
        "Existing review labels differ. "
        "Inspect the saved file before changing them."
    )
else:
    LABELS_PATH.write_text(
        json.dumps(
            annotations,
            ensure_ascii=False,
            indent=2
        ),
        encoding="utf-8"
    )

print("ROUND 1 HUMAN REVIEW RESULTS")
print("=" * 60)

for i, row in enumerate(annotations, 1):
    print(
        f"{i}. {row['id']}",
        "|", row["review_status"]
    )
    print("   ", row["finding"])

# ------------------------------------------------------------
# 2. Build provisional training/evaluation subsets
# ------------------------------------------------------------

train_quarantine = {
    row["id"]
    for row in annotations
    if row["review_status"] == "quarantine"
}

val_challenge = {
    row["id"]
    for row in annotations
    if row["review_status"] == "noisy_challenge"
}

original_train = selection["train_ids"]
original_val = selection["val_ids"]

assert train_quarantine.issubset(set(original_train))
assert val_challenge.issubset(set(original_val))

filtered_train = [
    rid for rid in original_train
    if rid not in train_quarantine
]

provisional_val = [
    rid for rid in original_val
    if rid not in val_challenge
]

verified_matches = {
    row["id"]
    for row in annotations
    if row["review_status"] == "verified_match"
}

filtered = {
    "status": "provisional_not_fully_reviewed",
    "train_ids": filtered_train,
    "primary_val_ids": provisional_val,
    "noisy_challenge_ids": sorted(val_challenge),
    "quarantined_train_ids": sorted(train_quarantine),
    "verified_match_ids": sorted(verified_matches),
    "source_selection": str(SELECTION_PATH),
    "note": (
        "Only five examples have undergone manual review. "
        "Unreviewed examples are not certified clean."
    )
}

if FILTERED_PATH.exists():
    previous = json.loads(
        FILTERED_PATH.read_text(encoding="utf-8")
    )
    assert previous == filtered, (
        "A different filtered split already exists."
    )
else:
    FILTERED_PATH.write_text(
        json.dumps(
            filtered,
            ensure_ascii=False,
            indent=2
        ),
        encoding="utf-8"
    )

print("\nPROVISIONAL DATASET")
print("=" * 60)

print("Original training:", len(original_train))
print("Quarantined training:", len(train_quarantine))
print("Remaining training:", len(filtered_train))

print("\nOriginal validation:", len(original_val))
print("Noisy challenge:", len(val_challenge))
print("Remaining primary validation:", len(provisional_val))

print("\nHuman-verified matching examples:",
      len(verified_matches))

print(
    "Other examples remain unreviewed:",
    len(filtered_train + provisional_val)
    - len(verified_matches)
)

# ------------------------------------------------------------
# 3. Select five more examples for human inspection
# ------------------------------------------------------------

already_reviewed = {
    row["id"] for row in annotations
}

training_pool = [
    row for row in audit["rows"]
    if row["id"] in filtered_train
    and row["id"] not in already_reviewed
]

validation_pool = [
    row for row in audit["rows"]
    if row["id"] in provisional_val
    and row["id"] not in already_reviewed
]

round2 = []
selected_ids = set()

def add_candidate(reason, row):
    if row["id"] not in selected_ids:
        round2.append({
            "id": row["id"],
            "reason": reason,
            "group": row["group"]
        })
        selected_ids.add(row["id"])

# Two training samples with low transcript density.
for row in sorted(
    training_pool,
    key=lambda r: r["chars_per_second"]
)[:2]:
    add_candidate(
        "Training: low transcript density",
        row
    )

# One high-loss training sample not already chosen.
for row in sorted(
    training_pool,
    key=lambda r: r["ctc_loss"],
    reverse=True
):
    if row["id"] not in selected_ids:
        add_candidate(
            "Training: high CTC loss",
            row
        )
        break

# One sparse validation example.
for row in sorted(
    validation_pool,
    key=lambda r: r["chars_per_second"]
):
    if row["id"] not in selected_ids:
        add_candidate(
            "Validation: low transcript density",
            row
        )
        break

# One high-loss validation example.
for row in sorted(
    validation_pool,
    key=lambda r: r["ctc_loss"],
    reverse=True
):
    if row["id"] not in selected_ids:
        add_candidate(
            "Validation: high CTC loss",
            row
        )
        break

assert len(round2) == 5

if ROUND2_PATH.exists():
    saved_round2 = json.loads(
        ROUND2_PATH.read_text(encoding="utf-8")
    )
    assert saved_round2 == round2
else:
    ROUND2_PATH.write_text(
        json.dumps(
            round2,
            ensure_ascii=False,
            indent=2
        ),
        encoding="utf-8"
    )

# ------------------------------------------------------------
# 4. Play the new recordings
# ------------------------------------------------------------

records_by_id = {
    record["id"]: record
    for record in train_records
}

print("\n" + "=" * 65)
print("ROUND 2: FIVE RECORDINGS TO REVIEW")
print("=" * 65)

for index, item in enumerate(round2, 1):

    rid = item["id"]
    row = audit_rows[rid]
    record = records_by_id[rid]

    audio_path = (
        Path(AUDIO_DIR) / record["audio_file"]
    )

    assert audio_path.is_file()

    print("\n" + "=" * 60)
    print(f"RECORDING {index}/5")
    print("=" * 60)

    print("ID:", rid)
    print("Reason:", item["reason"])
    print(
        "Duration:",
        round(row["audio_seconds"], 2),
        "seconds"
    )
    print(
        "CTC loss:",
        round(row["ctc_loss"], 3)
    )
    print(
        "Transcript density:",
        round(row["chars_per_second"], 2),
        "characters/second"
    )

    print("\nREFERENCE:")
    print(row["reference"])

    print("\nMODEL PREDICTION:")
    print(row["prediction"] or "[EMPTY]")

    print("\nAUDIO:")
    display(Audio(filename=str(audio_path)))

print("\n" + "=" * 65)
print("QUALITY AUDIT UPDATED")
print("=" * 65)

print("Annotations:", LABELS_PATH)
print("Filtered split:", FILTERED_PATH)
print("Next review queue:", ROUND2_PATH)
print("Optimizer updates: 0")
print("Existing checkpoints modified: 0")


## Original source cell 53

Position 54 in the original notebook; original code comments may use different numbering.


In [ ]:

# ============================================================
# CELL 49 - FINALIZE MANUAL QA AND CLEAN EVALUATION
#
# 4 matching recordings
# 1 recording with minor filler sounds
#
# Preserve original data and all checkpoints.
# ============================================================

import json
from pathlib import Path
from jiwer import cer, wer

QA_DIR = DATA_DIR / "scaled_amharic_64_16_v1"

ROUND2_QUEUE = QA_DIR / "manual_review_round2.json"
ROUND1_LABELS = QA_DIR / "manual_review_results_round1.json"
ORIGINAL_SPLIT = QA_DIR / "selection.json"
AUDIT_FILE = QA_DIR / "failure_mode_audit.json"
BASELINE_FILE = QA_DIR / "fp32_preflight.json"

ROUND2_RESULTS = QA_DIR / "manual_review_results_round2.json"
CLEAN_SPLIT_FILE = QA_DIR / "quality_filtered_v2.json"
CLEAN_METRICS_FILE = QA_DIR / "quality_metrics_v2.json"

def load_json(path):
    assert path.is_file(), f"Missing: {path}"
    return json.loads(path.read_text(encoding="utf-8"))

def save_checked(path, data):
    if path.exists():
        assert load_json(path) == data, (
            f"Existing file differs: {path}"
        )
        print("Already saved:", path.name)
    else:
        path.write_text(
            json.dumps(data, ensure_ascii=False, indent=2),
            encoding="utf-8"
        )
        print("Saved:", path.name)

queue = load_json(ROUND2_QUEUE)
round1 = load_json(ROUND1_LABELS)
original = load_json(ORIGINAL_SPLIT)
audit = load_json(AUDIT_FILE)
baseline = load_json(BASELINE_FILE)

assert len(queue) == 5

# ------------------------------------------------------------
# 1. Record the five new human assessments
# ------------------------------------------------------------

human_decisions = [
    ("verified_match", "Speech matches transcript"),
    ("verified_match", "Speech matches transcript"),
    (
        "match_with_fillers",
        "Correct speech with minor ah/hesitation sounds"
    ),
    ("verified_match", "Speech matches transcript"),
    ("verified_match", "Speech matches transcript"),
]

round2 = []

for item, (status, finding) in zip(
    queue, human_decisions
):
    round2.append({
        "id": item["id"],
        "original_group": item["group"],
        "review_status": status,
        "finding": finding
    })

assert len({r["id"] for r in round2}) == 5

save_checked(ROUND2_RESULTS, round2)

# ------------------------------------------------------------
# 2. Combine both review rounds
# ------------------------------------------------------------

all_reviews = round1 + round2

assert len(all_reviews) == 10
assert len({r["id"] for r in all_reviews}) == 10

quarantine = {
    r["id"]
    for r in all_reviews
    if r["review_status"] == "quarantine"
}

noisy_challenge = {
    r["id"]
    for r in all_reviews
    if r["review_status"] == "noisy_challenge"
}

verified = {
    r["id"]
    for r in all_reviews
    if r["review_status"] == "verified_match"
}

with_fillers = {
    r["id"]
    for r in all_reviews
    if r["review_status"] == "match_with_fillers"
}

assert len(quarantine) == 2
assert len(noisy_challenge) == 1
assert len(verified) == 6
assert len(with_fillers) == 1

# ------------------------------------------------------------
# 3. Construct the quality-filtered split
# ------------------------------------------------------------

train_ids = [
    rid for rid in original["train_ids"]
    if rid not in quarantine
]

val_ids = [
    rid for rid in original["val_ids"]
    if rid not in noisy_challenge
]

assert len(train_ids) == 62
assert len(val_ids) == 15
assert not set(train_ids) & set(val_ids)

records = {r["id"]: r for r in train_records}

train_speakers = {
    records[rid]["speaker_id"]
    for rid in train_ids
}

val_speakers = {
    records[rid]["speaker_id"]
    for rid in val_ids
}

assert not train_speakers & val_speakers

clean_split = {
    "status": "provisional_human_screened",
    "train_ids": train_ids,
    "primary_val_ids": val_ids,
    "quarantined_train_ids": sorted(quarantine),
    "noisy_challenge_ids": sorted(noisy_challenge),
    "verified_match_ids": sorted(verified),
    "match_with_fillers_ids": sorted(with_fillers),
    "reviewed_count": len(all_reviews),
    "note": (
        "Only 10 of the original 80 recordings were "
        "manually reviewed. Unreviewed recordings "
        "are not certified clean. Existing model "
        "checkpoints may already contain training "
        "influence from quarantined recordings."
    )
}

save_checked(CLEAN_SPLIT_FILE, clean_split)

# ------------------------------------------------------------
# 4. Recalculate validation metrics on the SAME 15 IDs
# ------------------------------------------------------------

baseline_by_id = {
    r["id"]: r
    for r in baseline["validation_predictions"]
}

pilot_by_id = {
    r["id"]: r
    for r in audit["rows"]
}

def compute_metrics(ids, source):
    refs = [source[rid]["reference"] for rid in ids]
    preds = [
        source[rid]["prediction"]
        for rid in ids
    ]

    return {
        "count": len(ids),
        "cer": float(cer(refs, preds)),
        "wer": float(wer(refs, preds)),
        "nonempty": sum(bool(p.strip()) for p in preds)
    }

assert all(rid in baseline_by_id for rid in val_ids)
assert all(rid in pilot_by_id for rid in val_ids)

before = compute_metrics(val_ids, baseline_by_id)
after = compute_metrics(val_ids, pilot_by_id)

challenge_ids = sorted(noisy_challenge)

challenge_before = compute_metrics(
    challenge_ids, baseline_by_id
)
challenge_after = compute_metrics(
    challenge_ids, pilot_by_id
)

train_after = compute_metrics(
    train_ids, pilot_by_id
)

metrics = {
    "primary_validation_before_pilot": before,
    "primary_validation_after_128_updates": after,
    "noisy_challenge_before": challenge_before,
    "noisy_challenge_after": challenge_after,
    "training_after_128_updates": train_after
}

save_checked(CLEAN_METRICS_FILE, metrics)

# ------------------------------------------------------------
# 5. Print summary
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("QUALITY REVIEW COMPLETE")
print("=" * 60)

print("Reviewed:", len(all_reviews))
print("Verified matches:", len(verified))
print("Matches with fillers:", len(with_fillers))
print("Quarantined:", len(quarantine))
print("Noisy challenge:", len(noisy_challenge))

print("\nFINAL PROVISIONAL DATASET")
print("Training:", len(train_ids))
print("Primary validation:", len(val_ids))
print("Noisy challenge:", len(challenge_ids))
print("Train/validation speaker overlap:", 0)

print("\n" + "=" * 60)
print("RECALCULATED METRICS")
print("=" * 60)

print(
    f"{'Dataset':<27}"
    f"{'CER':>10}"
    f"{'WER':>10}"
)

for label, result in [
    ("Validation before pilot", before),
    ("Validation after 128", after),
    ("Training after 128", train_after),
    ("Noisy before pilot", challenge_before),
    ("Noisy after 128", challenge_after)
]:
    print(
        f"{label:<27}"
        f"{result['cer']*100:>9.2f}%"
        f"{result['wer']*100:>9.2f}%"
    )

print("\nValidation CER improvement:",
      f"{(before['cer']-after['cer'])*100:+.2f}",
      "percentage points")

print("\nNo training was performed.")
print("All model checkpoints remain unchanged.")


## Original source cell 54

Position 55 in the original notebook; original code comments may use different numbering.


In [ ]:

# ============================================================
# CELL 50 - QUALITY-FILTERED TRAINING CONTINUATION
#
# 62 train / 15 primary validation
# Start from Cell 45 checkpoint, update 128
# Preserve optimizer state from previous training
# Train last 2 XLS-R layers + CTC head
#
# Three additional epochs (186 updates)
# FP32, no dropout/SpecAugment
# No previous checkpoints overwritten
# ============================================================

import json
import random
import hashlib
import statistics
import gc
from pathlib import Path

import torch
from transformers import Wav2Vec2ForCTC
from jiwer import cer, wer

ROOT_50 = DATA_DIR / "scaled_amharic_64_16_v1"
SOURCE_50 = ROOT_50 / "stage1_fp32" / "latest.pt"
SPLIT_50 = ROOT_50 / "quality_filtered_v2.json"
METRICS_50 = ROOT_50 / "quality_metrics_v2.json"

RUN_50 = ROOT_50 / "stage2_filtered_62_v1"
RUN_50.mkdir(parents=True, exist_ok=True)

LATEST_50 = RUN_50 / "latest.pt"
BEST_50 = RUN_50 / "best.pt"
HISTORY_50 = RUN_50 / "history.json"

DEVICE_50 = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

EPOCHS_50 = 3
HEAD_LR_50 = 1e-4
ENCODER_LR_50 = 2e-6
CLIP_50 = 1.0
SEED_50 = 20261020

def read_50(path):
    assert path.is_file(), f"Missing file: {path}"
    return json.loads(path.read_text(encoding="utf-8"))

split_50 = read_50(SPLIT_50)
previous_metrics_50 = read_50(METRICS_50)

train_ids_50 = split_50["train_ids"]
val_ids_50 = split_50["primary_val_ids"]

assert len(train_ids_50) == 62
assert len(val_ids_50) == 15
assert not (set(train_ids_50) & set(val_ids_50))

records_50 = {r["id"]: r for r in train_records}
indices_50 = {
    r["id"]: i for i, r in enumerate(train_dataset.records)
}

assert all(
    rid in indices_50
    for rid in train_ids_50 + val_ids_50
)

train_speakers_50 = {
    records_50[rid]["speaker_id"]
    for rid in train_ids_50
}
val_speakers_50 = {
    records_50[rid]["speaker_id"]
    for rid in val_ids_50
}

assert not (train_speakers_50 & val_speakers_50)

# Old examples versus newly introduced recordings.
original_8 = set(
    read_50(
        ROOT_50 / "selection.json"
    )["carryover_ids"]
)

old_ids_50 = [
    rid for rid in train_ids_50 if rid in original_8
]
new_ids_50 = [
    rid for rid in train_ids_50 if rid not in original_8
]

assert len(old_ids_50) == 7
assert len(new_ids_50) == 55

source_50 = torch.load(
    SOURCE_50,
    map_location="cpu",
    weights_only=True
)

assert source_50["step"] == 128
assert source_50["precision"] == "fp32"

configuration_50 = {
    "source_signature": source_50["signature"],
    "source_step": 128,
    "train_ids": train_ids_50,
    "val_ids": val_ids_50,
    "head_lr": HEAD_LR_50,
    "encoder_lr": ENCODER_LR_50,
    "clip": CLIP_50,
    "seed": SEED_50
}

signature_50 = hashlib.sha256(
    json.dumps(
        configuration_50,
        sort_keys=True,
        ensure_ascii=False
    ).encode("utf-8")
).hexdigest()

# ------------------------------------------------------------
# 1. Load original encoder, then restore saved fine-tuning
# ------------------------------------------------------------

print("RESTORING SCALED MODEL")
print("=" * 60)

model_50 = Wav2Vec2ForCTC.from_pretrained(
    str(BASELINE_DIR)
).to(DEVICE_50).float()

model_50.lm_head.load_state_dict(
    source_50["head"],
    strict=True
)

layers_50 = list(
    model_50.wav2vec2.encoder.layers[-2:]
)

for layer, weights in zip(
    layers_50,
    source_50["upper_layers"]
):
    layer.load_state_dict(weights, strict=True)

model_50.requires_grad_(False)

for layer in layers_50:
    layer.requires_grad_(True)

model_50.lm_head.requires_grad_(True)

# Keep stochastic dropout and SpecAugment disabled.
# Gradients still work normally.
model_50.eval()

encoder_parameters_50 = [
    p for layer in layers_50
    for p in layer.parameters()
    if p.requires_grad
]

head_parameters_50 = list(
    model_50.lm_head.parameters()
)

trainable_50 = (
    encoder_parameters_50 + head_parameters_50
)

assert sum(
    p.numel() for p in trainable_50
) == 25521473

optimizer_50 = torch.optim.AdamW([
    {
        "params": encoder_parameters_50,
        "lr": ENCODER_LR_50,
        "weight_decay": 0.01
    },
    {
        "params": head_parameters_50,
        "lr": HEAD_LR_50,
        "weight_decay": 0.0
    }
])

# Preserve accumulated optimizer moments.
optimizer_50.load_state_dict(
    source_50["optimizer"]
)

assert abs(
    optimizer_50.param_groups[0]["lr"] - ENCODER_LR_50
) < 1e-12

assert abs(
    optimizer_50.param_groups[1]["lr"] - HEAD_LR_50
) < 1e-12

# ------------------------------------------------------------
# 2. Restore stage 2 progress if present
# ------------------------------------------------------------

completed_50 = 0
history_50 = []
best_cer_50 = float(
    previous_metrics_50[
        "primary_validation_after_128_updates"
    ]["cer"]
)

if LATEST_50.exists():

    saved = torch.load(
        LATEST_50,
        map_location="cpu",
        weights_only=True
    )

    assert saved["signature"] == signature_50

    model_50.lm_head.load_state_dict(saved["head"])

    for layer, weights in zip(
        layers_50,
        saved["upper_layers"]
    ):
        layer.load_state_dict(weights)

    optimizer_50.load_state_dict(saved["optimizer"])

    completed_50 = int(saved["updates"])
    best_cer_50 = float(saved["best_cer"])

    if HISTORY_50.exists():
        history_50 = read_50(HISTORY_50)

    print("Resuming from stage 2 update:", completed_50)

else:
    print("Restored Cell 45 checkpoint at update 128.")

# ------------------------------------------------------------
# 3. Decode and evaluate recordings
# ------------------------------------------------------------

def batch_50(rid):

    item = train_dataset[indices_50[rid]]
    batch = ctc_collator([item])

    return {
        name: value.to(DEVICE_50)
        for name, value in batch.items()
    }

def evaluate_50(ids):

    model_50.eval()

    references = []
    predictions = []

    blank_frames = 0
    total_frames = 0

    with torch.no_grad():

        for rid in ids:

            batch = batch_50(rid)

            # Labels aren't needed for inference.
            output = model_50(
                input_values=batch["input_values"],
                attention_mask=batch["attention_mask"]
            )

            token_ids = output.logits.argmax(dim=-1)

            blank_frames += int(
                (token_ids == tokenizer.pad_token_id)
                .sum().item()
            )

            total_frames += token_ids.numel()

            prediction = processor.batch_decode(
                token_ids.cpu()
            )[0]

            prediction = normalize_amharic_ctc(
                prediction
            )

            reference = normalize_amharic_ctc(
                records_50[rid]["transcription"]
            )

            references.append(reference)
            predictions.append(prediction)

            del batch, output

    ref_chars = sum(
        len(text.replace(" ", ""))
        for text in references
    )

    pred_chars = sum(
        len(text.replace(" ", ""))
        for text in predictions
    )

    return {
        "cer": float(cer(references, predictions)),
        "wer": float(wer(references, predictions)),
        "length_ratio": pred_chars / max(ref_chars, 1),
        "blank_fraction": blank_frames / max(total_frames, 1),
        "nonempty": sum(bool(p.strip()) for p in predictions),
        "sample": predictions[0]
    }

def report_50(epoch, update):

    old = evaluate_50(old_ids_50)
    new = evaluate_50(new_ids_50)
    val = evaluate_50(val_ids_50)

    row = {
        "epoch": epoch,
        "updates": update,
        "old_train_cer": old["cer"],
        "new_train_cer": new["cer"],
        "val_cer": val["cer"],
        "val_wer": val["wer"],
        "val_length_ratio": val["length_ratio"],
        "val_blank_fraction": val["blank_fraction"],
        "val_nonempty": val["nonempty"],
        "val_sample": val["sample"]
    }

    print(f"\nEPOCH {epoch}/{EPOCHS_50}")
    print("-" * 58)

    print("Additional updates:", update)
    print(
        "Previous 7 training CER:",
        f"{old['cer']*100:.2f}%"
    )
    print(
        "New 55 training CER:",
        f"{new['cer']*100:.2f}%"
    )
    print(
        "Primary validation CER:",
        f"{val['cer']*100:.2f}%"
    )
    print(
        "Primary validation WER:",
        f"{val['wer']*100:.2f}%"
    )
    print(
        "Prediction/reference length:",
        f"{val['length_ratio']:.2f}"
    )
    print(
        "CTC blank frames:",
        f"{val['blank_fraction']*100:.1f}%"
    )
    print(
        "Nonempty validation predictions:",
        f"{val['nonempty']}/15"
    )

    print("Validation example:")
    print(val["sample"][:120] or "[EMPTY]")

    return row

def save_50(update, best_cer, save_best=False):

    checkpoint = {
        "signature": signature_50,
        "updates": update,
        "best_cer": best_cer,
        "precision": "fp32",
        "head": model_50.lm_head.state_dict(),
        "upper_layers": [
            layer.state_dict()
            for layer in layers_50
        ],
        "optimizer": optimizer_50.state_dict()
    }

    temp = RUN_50 / "latest.pt.partial"
    torch.save(checkpoint, temp)
    temp.replace(LATEST_50)

    if save_best:
        best_temp = RUN_50 / "best.pt.partial"
        torch.save(checkpoint, best_temp)
        best_temp.replace(BEST_50)

    hist_temp = RUN_50 / "history.json.partial"

    hist_temp.write_text(
        json.dumps(
            history_50,
            ensure_ascii=False,
            indent=2
        ),
        encoding="utf-8"
    )

    hist_temp.replace(HISTORY_50)

# ------------------------------------------------------------
# 4. Establish consistent starting metrics
# ------------------------------------------------------------

print("\nQUALITY-FILTERED TRAINING")
print("=" * 60)

print("Training recordings:", len(train_ids_50))
print("Primary validation:", len(val_ids_50))
print("Previously trained:", len(old_ids_50))
print("New training:", len(new_ids_50))
print("Trainable parameters:", 25521473)
print("FP32:", True)
print("Additional epochs:", EPOCHS_50)

if completed_50 == 0 and not history_50:

    starting_50 = report_50(0, 0)

    expected = previous_metrics_50[
        "primary_validation_after_128_updates"
    ]["cer"]

    assert abs(
        starting_50["val_cer"] - expected
    ) < 0.005, (
        "Starting validation CER differs from "
        "the saved Cell 49 baseline."
    )

    history_50.append(starting_50)

# ------------------------------------------------------------
# 5. Deterministic training order
# ------------------------------------------------------------

def training_id_50(update):

    index = update - 1
    epoch_index = index // len(train_ids_50)
    position = index % len(train_ids_50)

    order = list(train_ids_50)

    random.Random(
        SEED_50 + epoch_index
    ).shuffle(order)

    return order[position]

TOTAL_UPDATES_50 = EPOCHS_50 * len(train_ids_50)

losses_50 = []
norms_50 = []

# ------------------------------------------------------------
# 6. Run training
# ------------------------------------------------------------

for update in range(
    completed_50 + 1,
    TOTAL_UPDATES_50 + 1
):

    rid = training_id_50(update)

    optimizer_50.zero_grad(set_to_none=True)
    model_50.eval()

    batch = batch_50(rid)

    with torch.autocast(
        device_type=DEVICE_50.type,
        enabled=False
    ):
        loss = model_50(**batch).loss

    if not torch.isfinite(loss).item():
        raise RuntimeError(
            f"Non-finite loss at update {update}, "
            f"recording {rid}"
        )

    loss.backward()

    grad_norm = torch.nn.utils.clip_grad_norm_(
        trainable_50,
        max_norm=CLIP_50,
        error_if_nonfinite=True
    )

    raw_norm = float(grad_norm)

    if raw_norm > 5000:
        optimizer_50.zero_grad(set_to_none=True)
        raise RuntimeError(
            f"Extreme gradient norm at update "
            f"{update}, recording {rid}: {raw_norm:.1f}"
        )

    optimizer_50.step()

    losses_50.append(float(loss.item()))
    norms_50.append(raw_norm)

    completed_50 = update

    del batch, loss

    # Evaluate at the end of each full epoch.
    if update % len(train_ids_50) == 0:

        epoch = update // len(train_ids_50)

        row = report_50(epoch, update)

        row["mean_step_loss"] = statistics.mean(
            losses_50
        )
        row["median_gradient_norm"] = statistics.median(
            norms_50
        )
        row["max_gradient_norm"] = max(norms_50)
        row["clipped_fraction"] = (
            sum(n > CLIP_50 for n in norms_50)
            / len(norms_50)
        )

        improved = (
            row["val_cer"] < best_cer_50
        )

        if improved:
            best_cer_50 = row["val_cer"]

        history_50.append(row)

        save_50(
            update,
            best_cer_50,
            save_best=improved
        )

        print(
            "Mean training-step loss:",
            round(row["mean_step_loss"], 4)
        )
        print(
            "Median gradient norm:",
            round(row["median_gradient_norm"], 2)
        )
        print(
            "Maximum gradient norm:",
            round(row["max_gradient_norm"], 2)
        )
        print(
            "Fraction clipped:",
            f"{row['clipped_fraction']*100:.1f}%"
        )

        losses_50.clear()
        norms_50.clear()

# ------------------------------------------------------------
# 7. Final comparison
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("QUALITY-FILTERED TRAINING RESULTS")
print("=" * 70)

print(
    f"{'Epoch':<8}"
    f"{'Old train':<14}"
    f"{'New train':<14}"
    f"{'Val CER':<13}"
    f"{'Val WER':<13}"
    f"{'Length':<10}"
)

for row in history_50:

    print(
        f"{row['epoch']:<8}"
        f"{row['old_train_cer']*100:<14.2f}"
        f"{row['new_train_cer']*100:<14.2f}"
        f"{row['val_cer']*100:<13.2f}"
        f"{row['val_wer']*100:<13.2f}"
        f"{row['val_length_ratio']:<10.2f}"
    )

print("\nAdditional updates completed:", completed_50)
print("Best observed validation CER:",
      f"{best_cer_50*100:.2f}%")

print("Latest checkpoint:", LATEST_50)
print("History:", HISTORY_50)

if BEST_50.exists():
    print("Best improved checkpoint:", BEST_50)
else:
    print(
        "No improved stage-2 checkpoint; "
        "the stage-1 starting checkpoint remains best."
    )

print("All earlier model checkpoints unchanged.")


## Original source cell 55

Position 56 in the original notebook; original code comments may use different numbering.


In [ ]:

# ============================================================
# CELL 51 - AMHARIC CTC LABEL ROUND-TRIP AUDIT
#
# Verify that the labels used for training reconstruct
# the exact normalized reference transcripts.
#
# 62 training + 15 validation recordings.
# No training or checkpoint modifications.
# ============================================================

import json
import statistics
from collections import Counter
from pathlib import Path
from jiwer import cer

QA51_DIR = DATA_DIR / "scaled_amharic_64_16_v1"
QA51_SPLIT = QA51_DIR / "quality_filtered_v2.json"
QA51_OUTPUT = QA51_DIR / "label_roundtrip_audit.json"

selection = json.loads(
    QA51_SPLIT.read_text(encoding="utf-8")
)

groups = {
    "train": selection["train_ids"],
    "validation": selection["primary_val_ids"]
}

assert len(groups["train"]) == 62
assert len(groups["validation"]) == 15

record_lookup = {
    r["id"]: r for r in train_records
}

dataset_indices = {
    r["id"]: index
    for index, r in enumerate(train_dataset.records)
}

# XLS-R convolutional feature extractor configuration.
kernels = (10, 3, 3, 3, 2, 2, 2)
strides = (5, 2, 2, 2, 2, 2, 2)

# Read the actual architecture configuration instead
# of assuming its kernels/strides are correct.
from transformers import Wav2Vec2Config

config = Wav2Vec2Config.from_pretrained(str(BASELINE_DIR))
kernels = tuple(config.conv_kernel)
strides = tuple(config.conv_stride)

def output_frames(n_samples):
    length = int(n_samples)
    for kernel, stride in zip(kernels, strides):
        length = (length - kernel) // stride + 1
    return length

rows = []
token_counts = Counter()

print("AMHARIC CTC LABEL AUDIT")
print("=" * 60)
print("Tokenizer vocabulary size:", len(tokenizer))
print("CTC blank ID:", tokenizer.pad_token_id)
print("Unknown token ID:", tokenizer.unk_token_id)
print("Recording count:", 77)

for group_name, record_ids in groups.items():

    print(f"\nChecking {group_name}: {len(record_ids)} recordings")

    for rid in record_ids:

        record = record_lookup[rid]

        # Load the exact dataset example used in training.
        example = train_dataset[dataset_indices[rid]]
        batch = ctc_collator([example])

        # Remove only padding introduced by the collator.
        label_tensor = batch["labels"][0]
        label_ids = label_tensor[
            label_tensor != -100
        ].detach().cpu().tolist()

        # Decode WITHOUT collapsing repeated CTC symbols.
        # Repeated symbols in references must be preserved.
        decoded = tokenizer.decode(
            label_ids,
            group_tokens=False
        )

        reference = normalize_amharic_ctc(
            record["transcription"]
        )

        decoded = normalize_amharic_ctc(decoded)

        # Count unknown and blank target tokens.
        unknown_count = sum(
            tid == tokenizer.unk_token_id
            for tid in label_ids
        )

        blank_count = sum(
            tid == tokenizer.pad_token_id
            for tid in label_ids
        )

        repeated = sum(
            left == right
            for left, right in zip(
                label_ids[:-1], label_ids[1:]
            )
        )

        audio_samples = int(
            batch["attention_mask"][0].sum().item()
        )

        frames = output_frames(audio_samples)
        required_frames = len(label_ids) + repeated

        token_counts.update(label_ids)

        row = {
            "id": rid,
            "group": group_name,
            "reference": reference,
            "decoded_labels": decoded,
            "exact_match": reference == decoded,
            "roundtrip_cer": float(
                cer(reference, decoded)
            ),
            "label_length": len(label_ids),
            "unknown_count": unknown_count,
            "blank_target_count": blank_count,
            "audio_seconds": audio_samples / 16000,
            "available_frames": frames,
            "required_frames": required_frames,
            "ctc_feasible": frames >= required_frames
        }

        rows.append(row)

    print("Completed:", len(record_ids))

# ------------------------------------------------------------
# Summarize results
# ------------------------------------------------------------

mismatches = [
    r for r in rows
    if not r["exact_match"]
]

unknown_rows = [
    r for r in rows
    if r["unknown_count"] > 0
]

blank_rows = [
    r for r in rows
    if r["blank_target_count"] > 0
]

invalid_alignment = [
    r for r in rows
    if not r["ctc_feasible"]
]

references = [r["reference"] for r in rows]
decoded_all = [r["decoded_labels"] for r in rows]

print("\n" + "=" * 65)
print("LABEL INTEGRITY RESULTS")
print("=" * 65)

print("Recordings checked:", len(rows))
print("Exact transcript round-trips:",
      len(rows) - len(mismatches))
print("Transcript mismatches:", len(mismatches))
print("Recordings with unknown tokens:", len(unknown_rows))
print("Recordings with blank target tokens:", len(blank_rows))
print("CTC-infeasible recordings:", len(invalid_alignment))

print(
    "Overall label round-trip CER:",
    f"{cer(references, decoded_all)*100:.4f}%"
)

print(
    "Minimum CTC frame margin:",
    min(
        r["available_frames"] - r["required_frames"]
        for r in rows
    )
)

print(
    "Distinct token IDs appearing in labels:",
    len(token_counts)
)

# ------------------------------------------------------------
# Print any transcript reconstruction errors
# ------------------------------------------------------------

if mismatches:

    print("\nTRANSCRIPT MISMATCHES")
    print("=" * 65)

    for row in mismatches[:10]:

        print("\nID:", row["id"])
        print("Group:", row["group"])
        print("CER:", f"{row['roundtrip_cer']*100:.2f}%")
        print("REFERENCE:")
        print(row["reference"])
        print("DECODED LABELS:")
        print(row["decoded_labels"])

else:
    print("\nALL 77 LABELS RECONSTRUCT THEIR REFERENCES.")

# ------------------------------------------------------------
# Save full audit
# ------------------------------------------------------------

result = {
    "recordings_checked": len(rows),
    "exact_matches": len(rows) - len(mismatches),
    "mismatch_count": len(mismatches),
    "unknown_token_recordings": len(unknown_rows),
    "blank_target_recordings": len(blank_rows),
    "ctc_infeasible_recordings": len(invalid_alignment),
    "corpus_roundtrip_cer": float(
        cer(references, decoded_all)
    ),
    "rows": rows
}

QA51_OUTPUT.write_text(
    json.dumps(
        result,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print("\n" + "=" * 65)
print("AUDIT FINISHED")
print("=" * 65)
print("Saved:", QA51_OUTPUT)
print("Optimizer updates: 0")
print("Model checkpoints modified: 0")
